In [2]:
import customtkinter as ctk
import psycopg2
# Set global CustomTkinter appearance mode
ctk.set_appearance_mode("light")

# DATABASE CONNECTION

def connect():
    return psycopg2.connect(
        dbname="safepath",
        user="postgres",
        password="cos101",
        host="localhost",
        port="5432"
    )

# LIVE DATABASE FETCHING

def fetch_dashboard_metrics():

    conn = connect()
    cur = conn.cursor()

    # TOTAL REPORTS
    cur.execute("SELECT COUNT(*) FROM reports")
    total_reports = cur.fetchone()[0]

    # HIGH RISK REPORTS
    cur.execute("""
        SELECT COUNT(*)
        FROM reports
        WHERE urgency ILIKE 'High'
    """)
    high_risk = cur.fetchone()[0]

    # AWARENESS CHECKS
    # Placeholder logic for now
    awareness = total_reports + 5

    # PEOPLE HELPED
    # Placeholder estimation
    helped = total_reports

    # RECENT REPORTS
    cur.execute("""
        SELECT incident_type, location
        FROM reports
        ORDER BY id DESC
        LIMIT 5
    """)

    recent_rows = cur.fetchall()

    recent_reports = []

    for incident, location in recent_rows:
        recent_reports.append(
            f"{incident} reported in {location}"
        )

    cur.close()
    conn.close()

    return {
        "total": total_reports,
        "high_risk": high_risk,
        "awareness": awareness,
        "helped": helped,
        "recent": recent_reports
    }
# ==========================================
# APPLICATION MAIN WINDOW ARCHITECTURE
# ==========================================
root = ctk.CTk()
root.geometry("1440x900")  # Exact standard desktop design canvas size
root.title("SafePath Dashboard")

# Precise Figma Palette hex codes mapping
COLOR_SIDEBAR = "#4B1E0F"       # Deep rich brown
COLOR_WORKSPACE_BG = "#F3EBDD"   # Warm parchment cream background
COLOR_CARD_DARK = "#5A2D0C"      # Dark chocolate block
COLOR_CARD_ORANGE = "#B85C2E"    # High risk terracotta
COLOR_CARD_MID = "#6B3F1D"       # Medium brown status card
COLOR_CARD_LIGHT = "#D9C2A3"     # Light clay check card
COLOR_TEXT_DARK = "#3E2A1F"      # Deep primary label text

#SIDEBAR NAVIGATION

sidebar = ctk.CTkFrame(root, width=260, fg_color=COLOR_SIDEBAR, corner_radius=0)
sidebar.pack(side="left", fill="y")
sidebar.pack_propagate(False)

# App Logo Brand Label
logo = ctk.CTkLabel(sidebar, text="SafePath", font=("Georgia", 32, "bold"), text_color="white")
logo.pack(pady=(40, 30), anchor="w", padx=30)

# Navigation Menu Options
menu_items = ["Dashboard", "Situation Check", "Report Case", "Records", "Insights", "Resources", "Echoes of Home", "Settings"]
for item in menu_items:
    btn = ctk.CTkButton(
        sidebar, 
        text=item, 
        fg_color="#C47A3A" if item == "Dashboard" else "transparent",
        hover_color="#8A4F27",
        text_color="white", 
        font=("Arial", 14, "bold" if item == "Dashboard" else "normal"), 
        anchor="w", 
        height=40
    )
    btn.pack(fill="x", padx=20, pady=5)

tagline = ctk.CTkLabel(sidebar, text="Awareness. Protection. Freedom.", font=("Arial", 11, "italic"), text_color="#D9C2A3")
tagline.pack(side="bottom", pady=25)

# MAIN WORKSPACE CONTAINER

main_frame = ctk.CTkFrame(root, fg_color=COLOR_WORKSPACE_BG, corner_radius=0)
main_frame.pack(side="right", fill="both", expand=True)

# Welcome Header Titles
title = ctk.CTkLabel(main_frame, text="Dashboard", font=("Georgia", 38, "bold"), text_color=COLOR_TEXT_DARK)
title.pack(anchor="nw", padx=40, pady=(40, 5))

subtitle = ctk.CTkLabel(main_frame, text="Welcome back, James.", font=("Arial", 16), text_color="#5E4B3C")
subtitle.pack(anchor="nw", padx=40, pady=(0, 25))

# FLAT WORKSPACE LAYOUT GRID

workspace_grid = ctk.CTkFrame(main_frame, fg_color="transparent")
workspace_grid.pack(fill="both", expand=True, padx=40, pady=(0, 40))

# 2 Columns matching the balanced side-by-side design
workspace_grid.columnconfigure(0, weight=1)
workspace_grid.columnconfigure(1, weight=1)

# 3 Rows: Rows 0 & 1 hold the stats cards. Row 2 holds the major bottom panels.
workspace_grid.rowconfigure(0, weight=1)
workspace_grid.rowconfigure(1, weight=1)
workspace_grid.rowconfigure(2, weight=2)  # Extra height padding for bottom action content

# THE 2X2 STATS CARDS MATRIX


# Card 1: Total Reports [Row 0, Column 0]
total_card = ctk.CTkFrame(workspace_grid, fg_color=COLOR_CARD_DARK, corner_radius=18)
total_card.grid(row=0, column=0, sticky="nsew", padx=(0, 15), pady=10)
total_card.pack_propagate(False)
ctk.CTkLabel(total_card, text="Total Reports", font=("Arial", 16), text_color="#F3EBDD").pack(anchor="nw", padx=25, pady=(25, 5))
lbl_total_val = ctk.CTkLabel(total_card, text="--", font=("Georgia", 42, "bold"), text_color="white")
lbl_total_val.pack(anchor="nw", padx=25)

# Card 2: High Risk Flagged [Row 0, Column 1]
highrisk_card = ctk.CTkFrame(workspace_grid, fg_color=COLOR_CARD_ORANGE, corner_radius=18)
highrisk_card.grid(row=0, column=1, sticky="nsew", padx=(15, 0), pady=10)
highrisk_card.pack_propagate(False)
ctk.CTkLabel(highrisk_card, text="High Risk Flagged", font=("Arial", 16, "bold"), text_color=COLOR_SIDEBAR).pack(anchor="nw", padx=25, pady=(25, 5))
lbl_highrisk_val = ctk.CTkLabel(highrisk_card, text="--", font=("Georgia", 42, "bold"), text_color=COLOR_SIDEBAR)
lbl_highrisk_val.pack(anchor="nw", padx=25)

# Card 3: Awareness Check [Row 1, Column 0]
awareness_card = ctk.CTkFrame(workspace_grid, fg_color=COLOR_CARD_LIGHT, corner_radius=18)
awareness_card.grid(row=1, column=0, sticky="nsew", padx=(0, 15), pady=10)
awareness_card.pack_propagate(False)
ctk.CTkLabel(awareness_card, text="Awareness Check", font=("Arial", 16), text_color=COLOR_SIDEBAR).pack(anchor="nw", padx=25, pady=(25, 5))
lbl_awareness_val = ctk.CTkLabel(awareness_card, text="--", font=("Georgia", 42, "bold"), text_color=COLOR_SIDEBAR)
lbl_awareness_val.pack(anchor="nw", padx=25)

# Card 4: People Helped [Row 1, Column 1]
helped_card = ctk.CTkFrame(workspace_grid, fg_color=COLOR_CARD_MID, corner_radius=18)
helped_card.grid(row=1, column=1, sticky="nsew", padx=(15, 0), pady=10)
helped_card.pack_propagate(False)
ctk.CTkLabel(helped_card, text="People helped", font=("Arial", 16), text_color="#F3EBDD").pack(anchor="nw", padx=25, pady=(25, 5))
lbl_helped_val = ctk.CTkLabel(helped_card, text="--", font=("Georgia", 42, "bold"), text_color="white")
lbl_helped_val.pack(anchor="nw", padx=25)

 #THE BOTTOM RE-ALIGNED PANELS

# Panel Left: Situation / Risk Assessment CTA Module [Row 2, Column 0]
risk_assess_card = ctk.CTkFrame(workspace_grid, fg_color=COLOR_CARD_DARK, corner_radius=18)
risk_assess_card.grid(row=2, column=0, sticky="nsew", padx=(0, 15), pady=(15, 0))
risk_assess_card.pack_propagate(False)

ctk.CTkLabel(risk_assess_card, text="Unsure about a situation ?", font=("Georgia", 26, "bold"), text_color="#F3EBDD").pack(anchor="center", pady=(35, 10))
ctk.CTkLabel(risk_assess_card, text="Use our AI-driven analysis tool to identify signs\nof exploitation of forced labour.", font=("Arial", 15), text_color="#D9C2A3", justify="center").pack(anchor="center", pady=(0, 25))

btn_assess = ctk.CTkButton(
    risk_assess_card, 
    text="Start Risk Assessment", 
    fg_color=COLOR_CARD_ORANGE, 
    hover_color="#A44E25", 
    text_color="white", 
    font=("Arial", 15, "bold"), 
    height=45, 
    corner_radius=10
)
btn_assess.pack(anchor="center", ipadx=20)

# Panel Right: Recent Reports Feed Container [Row 2, Column 1]
recent_reports_card = ctk.CTkFrame(workspace_grid, fg_color="#EADFC9", corner_radius=18, border_width=1, border_color="#D9C2A3")
recent_reports_card.grid(row=2, column=1, sticky="nsew", padx=(15, 0), pady=(15, 0))
recent_reports_card.pack_propagate(False)

ctk.CTkLabel(recent_reports_card, text="Recent Reports", font=("Georgia", 22, "bold"), text_color=COLOR_TEXT_DARK).pack(anchor="nw", padx=25, pady=(25, 10))

# Clean visual box inside to display recent items list cleanly
feed_container = ctk.CTkFrame(recent_reports_card, fg_color="transparent")
feed_container.pack(fill="both", expand=True, padx=25)

# AUTOMATIC LIVE DATA REFRESH FUNCTION

def update_dashboard_view():

    data = fetch_dashboard_metrics()

    lbl_total_val.configure(text=data["total"])
    lbl_highrisk_val.configure(text=data["high_risk"])
    lbl_awareness_val.configure(text=data["awareness"])
    lbl_helped_val.configure(text=data["helped"])

    for widget in feed_container.winfo_children():
        widget.destroy()

    if data["recent"]:

        for report_text in data["recent"]:

            item_lbl = ctk.CTkLabel(
                feed_container,
                text=f"• {report_text}",
                font=("Arial", 14),
                text_color=COLOR_TEXT_DARK,
                anchor="w"
            )

            item_lbl.pack(fill="x", pady=6)

            ctk.CTkFrame(
                feed_container,
                height=1,
                fg_color="#D9C2A3"
            ).pack(fill="x", pady=(2, 6))

    else:

        ctk.CTkLabel(
            feed_container,
            text="No new notifications found.",
            font=("Arial", 14, "italic"),
            text_color="#766555"
        ).pack(pady=20)

    # AUTO REFRESH EVERY 5 SECONDS
    root.after(5000, update_dashboard_view)
# Initial draw execution
update_dashboard_view()

root.mainloop()

IndentationError: unexpected indent (2112252454.py, line 176)